In [ ]:
from telethon import TelegramClient
import asyncio
from datetime import timezone,datetime
import os
import pandas as pd
import pickle
import time
from cleantext import clean
import re
import calendar
from os import listdir
from os.path import isfile, join
from ast import literal_eval

import spacy
import nltk
from nltk.stem.porter import *      #rip spacy did not have Porter Stemmer

from sklearn.model_selection import train_test_split
from scipy.stats import mannwhitneyu
from scipy.stats import kruskal

import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from Crypto.Cipher import AES

In [ ]:
#enhance the model I am using first
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

with open ("data\\texttest.ob", "rb") as fp:
    sentences=pickle.load(fp)

#testing only
model_dir="./bert_binary_final"
tokenizer=BertTokenizer.from_pretrained(model_dir)
model=BertForSequenceClassification.from_pretrained(model_dir)
# Move model to appropriate device (e.g., GPU if available)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Set to evaluation mode

for id,things in enumerate(sentences):
    sentences[id]=str(things)

# Define batch size (adjust based on your GPU; start small)
batch_size = 8  # Experiment with 8, 16, or 32

# Process in batches
predictions = []
for i in range(0, len(sentences), batch_size):
    batch_sentences = sentences[i:i + batch_size]
    
    # Tokenize batch
    inputs = tokenizer(
        batch_sentences,
        padding="max_length",
        truncation=True,
        max_length=100,  # Already reasonable; could lower to 64 if needed
        return_tensors="pt"
    )
    
    # Move inputs to GPU
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # Predict with no gradients
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()  # Move to CPU immediately
    
    # Store predictions
    predictions.extend(batch_preds)
    
    # Clear GPU memory for this batch
    del inputs, outputs, logits, batch_preds
    torch.cuda.empty_cache()

# Print results
for sent, pred in zip(sentences, predictions):
    print(f"Sentence: '{sent}' -> Predicted Label: {pred}")

In [4]:
predictions[18]=0
predictions[23]=0
predictions[30]=0
predictions[40]=1
predictions[42]=1
predictions[55]=0
predictions[60]=0
predictions[76]=0
predictions[79]=1
predictions[80]=0
predictions[81]=1
predictions[92]=1
predictions[93]=0
predictions[108]=0
predictions[135]=0
predictions[139]=0
predictions[141]=0
predictions[148]=0
predictions[150]=0
predictions[166]=0
predictions[169]=0
predictions[170]=1
predictions[206]=1
predictions[210]=1
predictions[212]=0
predictions[225]=0
predictions[246]=1
predictions[253]=0
predictions[255]=1
predictions[260]=0
predictions[268]=1
predictions[270]=0
predictions[285]=0
predictions[293]=1
predictions[301]=0
predictions[310]=0
predictions[316]=1
predictions[317]=0
predictions[340]=0
predictions[358]=0
predictions[361]=0
predictions[366]=0
predictions[387]=0
predictions[392]=0
predictions[407]=0
predictions[419]=0
predictions[430]=0
predictions[435]=0
predictions[439]=0
predictions[443]=0
predictions[451]=1
predictions[454]=1
predictions[456]=0
predictions[475]=1
predictions[484]=0
predictions[492]=0
predictions[493]=1
predictions[494]=1
predictions[495]=1
predictions[502]=0
predictions[508]=0
predictions[515]=0
predictions[520]=1
predictions[525]=0
predictions[529]=0
predictions[532]=0
predictions[535]=0
predictions[541]=1
predictions[544]=0
predictions[565]=0
predictions[567]=0
predictions[584]=0
predictions[595]=0
predictions[599]=0
predictions[610]=1
predictions[611]=0
predictions[617]=1
predictions[622]=0
predictions[628]=0
predictions[638]=1
predictions[643]=0
predictions[646]=0
predictions[655]=0
predictions[657]=0
predictions[677]=0
predictions[678]=1
predictions[679]=0
predictions[689]=0
predictions[694]=0
predictions[707]=0
predictions[712]=0
predictions[745]=0
predictions[761]=0
predictions[763]=0
predictions[764]=0
predictions[766]=0
predictions[800]=0
predictions[807]=1
predictions[816]=0
predictions[817]=0
predictions[826]=1
predictions[827]=1
predictions[829]=1
predictions[837]=0
predictions[845]=0
predictions[851]=0
predictions[859]=0
predictions[868]=0
predictions[885]=0
predictions[890]=0
predictions[909]=0
predictions[921]=0
predictions[924]=0
predictions[935]=1
predictions[952]=1
predictions[955]=1
predictions[969]=0
predictions[976]=0
predictions[995]=1
predictions[999]=0
predictions[1000]=0
predictions[1002]=1
predictions[1008]=0
predictions[1012]=0
predictions[1019]=1
predictions[1023]=1
predictions[1026]=1
predictions[1028]=0

In [ ]:
#Model training
#Run this only when you don't have a trained model. Skip it if you have one
torch.cuda.empty_cache()
testsentence=[
    "ga come clean ma house",
    "augur no ch",
    "wolf, eat miku",
    "aw no act. can't kill miku",
    "lowly",
    "is the ota wc or dg?",
    "ty cupid 😘😘",
    "nice drunk wolf",
    "i'm safe by seer",
    "oh really mason",
    "blind wolf",
    "ota"
]

if (os.path.isdir("bert_binary_v2final")):
    raise Exception("Model already exist")

# Sample data (small dataset with labels)
train_sentences=sentences
train_labels=predictions  # Binary labels: 1 = positive, 0 = negative
test_sentences=testsentence  # Test set

# Step 1: Prepare dataset
train_data={"text": train_sentences, "label": train_labels}
test_data={"text": test_sentences}
train_dataset=Dataset.from_dict(train_data)
test_dataset=Dataset.from_dict(test_data)

# Step 2: Load tokenizer and model
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")
model=BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)  # 2 classes
device=torch.device("cuda" if torch.cuda.is_available() else "cpu") #should use GPU instead of CPU
model.to(device)
print(f"Model is on: {next(model.parameters()).device}")

# Step 3: Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

# Tokenize datasets
tokenized_train=train_dataset.map(tokenize_function, batched=True)
tokenized_test=test_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns and set format for PyTorch
tokenized_train=tokenized_train.remove_columns(["text"]).rename_column("label", "labels")
tokenized_test=tokenized_test.remove_columns(["text"])  # No labels in test for prediction
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask"])

# Step 4: Define compute_metrics for evaluation (optional)
def compute_metrics(pred):
    labels=pred.label_ids
    preds=np.argmax(pred.predictions, axis=1)
    accuracy=accuracy_score(labels, preds)
    precision, recall, f1, _=precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# Step 5: Training arguments
training_args=TrainingArguments(
    output_dir="./bert_binary",
    per_device_train_batch_size=2,  # Small batch for small dataset
    num_train_epochs=5,  # Few epochs to avoid overfitting
    logging_steps=40,
    save_steps=800,
    evaluation_strategy="no",  # No eval set here; add if you split data
    learning_rate=2e-5,  # Standard for BERT fine-tuning
    weight_decay=0.01,  # Regularization
)

# Step 6: Initialize Trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    compute_metrics=compute_metrics,  # Optional
)

# Step 7: Train the model
trainer.train(resume_from_checkpoint=True)

# Step 8: Predict on test set
predictions=trainer.predict(tokenized_test)
pred_labels=np.argmax(predictions.predictions, axis=1)  # 0 or 1

# Output predictions
print("Test Predictions (0/1):", pred_labels.tolist())
for sent, pred in zip(test_sentences, pred_labels):
    print(f"Sentence: '{sent}' -> Predicted Label: {pred}")


output_dir="./bert_binary_v2final"  # Directory to save the model
# Save the model
trainer.save_model(output_dir)
# Save the tokenizer
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")

In [7]:
with open ("data\\texttrain.ob", "rb") as fp:
    testsentences=pickle.load(fp)

label=[0,1,0,1,0,1,0,0,0,1,
       1,1,1,1,0,1,1,1,1,1,
       1,1,0,1,1,1,1,1,1,1,
       0,1,0,0,1,1,1,0,1,1,
       1,1,0,1,1,1,0,1,1,0,
       1,1,0,1,0,0,1,0,1,0,
       1,1,0,0,0,1,1,1,0,0,
       0,0,0,1,0,0,0,1,0,1,
       1,1,1,0,1,1,1,1,0,1,
       1,1,0,1,0,1,1,0,1,0,
       1,0,1,1,0,0,1,0,1,1,
       1,1,0,1,1,1,1,0,0,0,
       1,0,0,1,0,0,1,1,1,1,
       0,0,1,1,0,0,1,1,1,0,
       0,1,1,0,0,0,1,0,1,1,
       1,1,1,0,0,0,1,1,1,0,
       1,1,0,0,1,1,1,0,1,1,
       1,0,1,0,0,1,0,0,1,1,
       1,0,0,1,0,1,1,0,0,0,
       1,0,1,1,0,1,1,1,1,0,
       1,1,1,1,1,1,1,0,1,0,
       0,1,1,0,1,1,1,1,0,1,
       1,0,1,0,0,1,1,0,0,1,
       1,0,0,1,0,0,0,1,1,1,
       1,0,0,1,0,0,0,1,0,1,
       1,1,0,1,0,1,1,1]    #label it 0 or 1 for the training set,258 total,0=not talking roles about user themselves,1=talking roles about themselves

testsentences+=["ota","me ota","ota is wc?","ota pls don't kill","any ota","ota here"] #adding ota to the list for ota detection
label+=[1,1,0,1,0,1]

In [13]:
#testing only
model_dir="./bert_binary_v2final"
tokenizer=BertTokenizer.from_pretrained(model_dir)
model=BertForSequenceClassification.from_pretrained(model_dir)
# Move model to appropriate device (e.g., GPU if available)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Set to evaluation mode

inputs=tokenizer(
    testsentences,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
# Move inputs to device
inputs={key: val.to(device) for key, val in inputs.items()}

# Predict
with torch.no_grad():
    outputs=model(**inputs)
    logits=outputs.logits
    predictions=torch.argmax(logits, dim=-1).cpu().numpy()  # 0 or 1

for sent, pred in zip(testsentences, predictions):
    print(f"Sentence: '{sent}' -> Predicted Label: {pred}")

Sentence: 'number 2 u r gunner' -> Predicted Label: 0
Sentence: 'no act again, dont waste ur act on me' -> Predicted Label: 1
Sentence: 'wait for mason.' -> Predicted Label: 0
Sentence: 'no act not lovers' -> Predicted Label: 1
Sentence: 'arso game huh' -> Predicted Label: 0
Sentence: 'augur no dg' -> Predicted Label: 1
Sentence: 'poor paci burden' -> Predicted Label: 0
Sentence: 'traitor are seen as ww' -> Predicted Label: 0
Sentence: 'idk but this ww like weak' -> Predicted Label: 0
Sentence: 'i'm not the wolf anyway.' -> Predicted Label: 1
Sentence: 'me gd' -> Predicted Label: 1
Sentence: 'oh wait im chem' -> Predicted Label: 1
Sentence: 'gd 🫡' -> Predicted Label: 1
Sentence: 'am augur' -> Predicted Label: 1
Sentence: 'the oracle is suspicious though.' -> Predicted Label: 0
Sentence: 'im no egg' -> Predicted Label: 1
Sentence: 'oracle' -> Predicted Label: 1
Sentence: 'oh yea tm' -> Predicted Label: 1
Sentence: 'mason/vegetable' -> Predicted Label: 1
Sentence: 'i'm oracle btw' -> Pre

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

accuracy = accuracy_score(label, predictions)
precision = precision_score(label, predictions, average='binary')  # Use 'binary' for binary classification
recall = recall_score(label, predictions, average='binary')
f1 = f1_score(label, predictions, average='binary')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Detailed report
print("\nClassification Report:")
print(classification_report(label, predictions))

Accuracy: 0.9015
Precision: 0.9177
Recall: 0.9177
F1-Score: 0.9177

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88       106
           1       0.92      0.92      0.92       158

    accuracy                           0.90       264
   macro avg       0.90      0.90      0.90       264
weighted avg       0.90      0.90      0.90       264

